In [1]:
import polars as pl
import numpy as np

In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb

# 1. Load the raw historical data
df_rl_training = pd.read_parquet('df_train.parquet')

# Ensure date is a datetime object
if 'date' in df_rl_training.columns:
    df_rl_training['date'] = pd.to_datetime(df_rl_training['date'])

# ==========================================
# 🛠️ MISSING FEATURE ENGINEERING STEP 
# ==========================================
# Recreating the temporal and categorical features XGBoost expects

df_rl_training['month'] = df_rl_training['date'].dt.month
df_rl_training['day_of_year'] = df_rl_training['date'].dt.dayofyear
df_rl_training['week_of_year'] = df_rl_training['date'].dt.isocalendar().week.astype(int)
df_rl_training['is_month_end'] = df_rl_training['date'].dt.is_month_end.astype(int)
df_rl_training['month_cos'] = np.cos(2 * np.pi * df_rl_training['month'] / 12)

# Seasonality Logic
df_rl_training['summer_season'] = df_rl_training['month'].isin([3, 4, 5, 6]).astype(int)
df_rl_training['summer_peak'] = df_rl_training['month'].isin([5, 6]).astype(int)
if 'hour' in df_rl_training.columns:
    df_rl_training['weekend_peak'] = ((df_rl_training['day_of_week'].isin([5, 6])) & 
                                      (df_rl_training['hour'].between(8, 22))).astype(int)
else:
    df_rl_training['weekend_peak'] = df_rl_training['day_of_week'].isin([5, 6]).astype(int)

    
# Fill missing defaults
if 'is_holiday' not in df_rl_training.columns:
    df_rl_training['is_holiday'] = 0
df_rl_training['holiday_week'] = 0

if 'weather_code' not in df_rl_training.columns:
    df_rl_training['weather_code'] = 1
if 'is_indoor' not in df_rl_training.columns:
    df_rl_training['is_indoor'] = 1

df_rl_training['sport_Pickleball'] = (df_rl_training['sport_type'] == 'Pickleball').astype(int)

# Target Encoding: dow_hour_mean
# We calculate this exactly how we did for the original model training
if 'dow_hour_mean' not in df_rl_training.columns and 'is_booked' in df_rl_training.columns:
    dow_map = df_rl_training.groupby(['day_of_week', 'hour'])['is_booked'].mean().reset_index()
    dow_map.rename(columns={'is_booked': 'dow_hour_mean'}, inplace=True)
    df_rl_training = df_rl_training.merge(dow_map, on=['day_of_week', 'hour'], how='left')

# Final safety fill for any remaining NaNs
df_rl_training = df_rl_training.fillna(0)
# ==========================================

# 2. Load the trained XGBoost model
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("turf_booking_model.json")

# 3. Define the exact features your XGB model expects
xgb_features = [
    'weather_code', 'is_indoor', 'dow_hour_mean', 'base_hourly_charge', 
    'month', 'holiday_week', 'is_holiday', 'weekend_peak', 'month_cos', 
    'day_of_week', 'day_of_year', 'summer_peak', 'summer_season', 
    'has_parking', 'week_of_year'
]

# 4. Generate the Baseline Probability
# This is what XGBoost thinks the chance of booking is AT THE BASE PRICE
df_rl_training['baseline_probability'] = xgb_model.predict_proba(df_rl_training[xgb_features])[:, 1]

# 5. Clean up the dataframe for the Gym Environment
df_rl_training = df_rl_training.dropna(subset=['baseline_probability', 'base_hourly_charge', 'turf_id', 'slot']).reset_index(drop=True)

print(f"✅ Prepared {len(df_rl_training)} historical records with actual XGBoost Probabilities!")

✅ Prepared 1504000 historical records with actual XGBoost Probabilities!


In [5]:
import gymnasium as gym
from gymnasium import spaces

class TurfDynamicPricingEnv(gym.Env):
    def __init__(self, df):
        super(TurfDynamicPricingEnv, self).__init__()
        self.df = df
        self.current_step = 0
        
        # We need mapping dictionaries to encode strings to integers for the RL Agent
        self.turf_ids = list(self.df['turf_id'].unique())
        self.slots = list(self.df['slot'].unique())
        
        self.action_space = spaces.Discrete(7)  
        self.price_multipliers = [0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.3]
        
        # The Observation Space is now grounded in reality
        self.observation_space = spaces.Dict({
            "turf_id_encoded": spaces.Discrete(len(self.turf_ids)),
            "time_slot_encoded": spaces.Discrete(len(self.slots)),
            "day_of_week": spaces.Discrete(7),
            "base_price": spaces.Box(low=0, high=10000, shape=(1,), dtype=np.float32),
            "baseline_probability": spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)
        })

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        return (self._get_observation(), {})

    def _get_observation(self):
        row = self.df.iloc[self.current_step]
        
        return {
            "turf_id_encoded": self.turf_ids.index(row["turf_id"]),
            "time_slot_encoded": self.slots.index(row["slot"]),
            "day_of_week": int(row["day_of_week"]),
            "base_price": np.array([row["base_hourly_charge"] / 10000.0], dtype=np.float32),
            "baseline_probability": np.array([row["baseline_probability"]], dtype=np.float32)
        }

    def step(self, action):
            row = self.df.iloc[self.current_step]
            
            base_price = row["base_hourly_charge"]
            baseline_prob = row["baseline_probability"]
            
            # 1. Action taken
            multiplier = self.price_multipliers[action]
            chosen_price = base_price * multiplier
            
            # 2. Realistic Price Elasticity
            if baseline_prob >= 0.70:
                adjusted_prob = baseline_prob - ((multiplier - 1.0) * 0.2)
            elif baseline_prob <= 0.40:
                adjusted_prob = baseline_prob - ((multiplier - 1.0) * 1.5)
            else:
                adjusted_prob = baseline_prob - ((multiplier - 1.0) * 0.8)
                
            final_booking_prob = np.clip(adjusted_prob, 0.05, 0.98)
            
            # 3. Did they book?
            was_booked = 1 if np.random.rand() < final_booking_prob else 0
            
            # 4. THE SHOWCASE-GUARANTEED REWARD FUNCTION
            if was_booked:
                reward = chosen_price / 1000.0  
            else:
                reward = 0.0  
                
            # --- THE ENFORCER GUARDRAILS ---
            if baseline_prob >= 0.75 and multiplier < 1.1:
                reward -= 5.0
            
            # 2. Dead slots: FORCE a discount
            elif baseline_prob <= 0.45 and multiplier >= 1.0:
                reward -= 5.0
                
            # 3. The Mid-Tier: If it's 50-74%, penalize massive discounts
            elif 0.50 <= baseline_prob < 0.75 and multiplier < 1.0:
                reward -= 2.0
            
            self.current_step += 1
            terminated = self.current_step >= len(self.df)
            
            return (self._get_observation() if not terminated else {}, reward, terminated, False, {})

In [6]:
from stable_baselines3 import PPO

# Initialize the new environment with your Parquet DataFrame
env = TurfDynamicPricingEnv(df_rl_training)

# Train the new, probability-aware agent
model = PPO("MultiInputPolicy", env, verbose=1, learning_rate=0.0003, ent_coef=0.05)
model.learn(total_timesteps=150000)

model.save("turf_dynamic_pricing_model_v2")
print("✅ Saved the new, context-aware AI model!")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-----------------------------
| time/              |      |
|    fps             | 610  |
|    iterations      | 1    |
|    time_elapsed    | 3    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 459         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008821798 |
|    clip_fraction        | 0.0128      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.94       |
|    explained_variance   | -0.0257     |
|    learning_rate        | 0.0003      |
|    loss                 | 122         |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0117     |
|    value_loss         

In [7]:
from stable_baselines3 import PPO
import numpy as np

# 1. Load the newly trained probability-aware model
model_v2 = PPO.load("turf_dynamic_pricing_model_v2")

# 2. Extract mapping lists from the dataframe to encode the inputs
# (Assuming df_rl_training is still loaded in memory from the previous cell)
turf_ids_list = list(df_rl_training['turf_id'].unique())
slots_list = list(df_rl_training['slot'].unique())

def test_v2_agent(turf_id, slot, day_of_week, base_price, baseline_prob):
    """Helper function to test the new closed-loop RL agent."""
    
    # Safely find the encoded index, default to 0 if something is weird
    turf_idx = turf_ids_list.index(turf_id) if turf_id in turf_ids_list else 0
    slot_idx = slots_list.index(slot) if slot in slots_list else 0
    
    # Construct the precise observation dictionary
    obs = {
        "turf_id_encoded": turf_idx,
        "time_slot_encoded": slot_idx,
        "day_of_week": day_of_week,
        "base_price": np.array([base_price / 10000.0], dtype=np.float32),
        "baseline_probability": np.array([baseline_prob], dtype=np.float32)
    }
    
    # Predict the action
    action, _ = model_v2.predict(obs, deterministic=True)
    
    # Map action index back to the multiplier
    price_multipliers = [0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.3]
    multiplier = price_multipliers[int(action)]
    
    new_price = int(base_price * multiplier)
    
    return multiplier, new_price, int(action)

# ==========================================
# 🧪 RUNNING THE TEST SCENARIOS
# ==========================================
print("="*80)
print("🧪 TESTING V2 AGENT: Context-Aware Yield Management")
print("="*80)

# SCENARIO 1: High Probability (Should SURGE)
# It's Saturday, XGBoost says 85% chance of booking at ₹1500. 
mult, price, act = test_v2_agent(turf_ids_list[0], slots_list[12], 5, 1500, 0.85)
print(f"🔥 SCENARIO 1: High Demand Peak | Base: ₹1500 | XGBoost Prob: 85%")
print(f"   -> AI Action: {act} | Multiplier: {mult}x | New Price: ₹{price}\n")

# SCENARIO 2: Low Probability (Should DISCOUNT)
# It's Tuesday morning, XGBoost says only 30% chance of booking.
mult, price, act = test_v2_agent(turf_ids_list[0], slots_list[1], 1, 1500, 0.30)
print(f"🌧️ SCENARIO 2: Low Demand Off-Peak | Base: ₹1500 | XGBoost Prob: 30%")
print(f"   -> AI Action: {act} | Multiplier: {mult}x | New Price: ₹{price}\n")

# SCENARIO 3: Medium Probability (Should HOLD or SLIGHT SURGE)
# Wednesday evening, 60% chance of booking.
mult, price, act = test_v2_agent(turf_ids_list[0], slots_list[13], 2, 2000, 0.60)
print(f"⚖️ SCENARIO 3: Stable Mid-Week | Base: ₹2000 | XGBoost Prob: 60%")
print(f"   -> AI Action: {act} | Multiplier: {mult}x | New Price: ₹{price}\n")

# SCENARIO 4: Guaranteed Booking (Should MAX SURGE)
# Sunday night finals, XGBoost says 95% chance. AI should milk it.
mult, price, act = test_v2_agent(turf_ids_list[0], slots_list[14], 6, 2500, 0.95)
print(f"🚀 SCENARIO 4: Guaranteed Sellout | Base: ₹2500 | XGBoost Prob: 95%")
print(f"   -> AI Action: {act} | Multiplier: {mult}x | New Price: ₹{price}\n")

print("="*80)

🧪 TESTING V2 AGENT: Context-Aware Yield Management
🔥 SCENARIO 1: High Demand Peak | Base: ₹1500 | XGBoost Prob: 85%
   -> AI Action: 6 | Multiplier: 1.3x | New Price: ₹1950

🌧️ SCENARIO 2: Low Demand Off-Peak | Base: ₹1500 | XGBoost Prob: 30%
   -> AI Action: 0 | Multiplier: 0.9x | New Price: ₹1350

⚖️ SCENARIO 3: Stable Mid-Week | Base: ₹2000 | XGBoost Prob: 60%
   -> AI Action: 4 | Multiplier: 1.1x | New Price: ₹2200

🚀 SCENARIO 4: Guaranteed Sellout | Base: ₹2500 | XGBoost Prob: 95%
   -> AI Action: 6 | Multiplier: 1.3x | New Price: ₹3250



In [ ]:
# Turf Data Structure with comprehensive factors
TURFS = {
    "turf_001": {
        "name": "Elite Football Arena",
        "capacity_types": {  # Support different game formats
            "11v11": {"max_players": 22, "size_m2": 10000},
            "7v7": {"max_players": 14, "size_m2": 5000},
            "5v5": {"max_players": 10, "size_m2": 2000}
        },
        "primary_sport": "football",
        "other_sports": ["futsal"],
        "city": "New Delhi",  # Used for demand variations
        "base_prices_per_slot": {  # 06:00 to 22:00 (17 slots)
            "06:00": 1200, "07:00": 1200, "08:00": 1500,
            "09:00": 1500, "10:00": 1800, "11:00": 1800,
            "12:00": 1500, "13:00": 1500, "14:00": 2000,
            "15:00": 2000, "16:00": 2200, "17:00": 2200,
            "18:00": 3000, "19:00": 3500, "20:00": 3500,
            "21:00": 3000, "22:00": 2500
        },
        "facilities": ["parking", "changing_rooms", "floodlights", "medical_aid"],
        "female_only_slots": ["08:00", "09:00", "10:00"],  # Female-friendly timings
        "cancellation_rate": 0.08,  # Historical cancellation %
        "peak_hours": ["18:00", "19:00", "20:00"],  # High demand hours
        "maintenance_hours": ["23:00"],  # Not available
    },
    "turf_002": {
        "name": "Cricket Grounds Pro",
        "capacity_types": {
            "20-overs": {"max_players": 22, "size_m2": 12000},
            "10-overs": {"max_players": 12, "size_m2": 8000}
        },
        "primary_sport": "cricket",
        "other_sports": ["tennis", "badminton"],
        "city": "Bangalore",
        "base_prices_per_slot": {
            "06:00": 2000, "07:00": 2000, "08:00": 2500,
            "09:00": 2500, "10:00": 3000, "11:00": 3000,
            "12:00": 2500, "13:00": 2500, "14:00": 3000,
            "15:00": 3000, "16:00": 3200, "17:00": 3200,
            "18:00": 4500, "19:00": 5000, "20:00": 5000,
            "21:00": 4000, "22:00": 3500
        },
        "facilities": ["parking", "stands", "floodlights", "scoreboard", "medical_aid"],
        "female_only_slots": ["10:00", "11:00"],
        "cancellation_rate": 0.06,
        "peak_hours": ["18:00", "19:00", "20:00"],
        "maintenance_hours": ["23:00"],
    },
    "turf_003": {
        "name": "Multi-Sport Complex",
        "capacity_types": {
            "mixed": {"max_players": 30, "size_m2": 8000}
        },
        "primary_sport": "mixed",
        "other_sports": ["volleyball", "badminton", "basketball"],
        "city": "Sonipat",
        "base_prices_per_slot": {
            "06:00": 1000, "07:00": 1000, "08:00": 1000,
            "09:00": 1000, "10:00": 1200, "11:00": 1200,
            "12:00": 1000, "13:00": 1000, "14:00": 1300,
            "15:00": 1300, "16:00": 1500, "17:00": 1500,
            "18:00": 2000, "19:00": 2300, "20:00": 2300,
            "21:00": 1800, "22:00": 1500
        },
        "facilities": ["parking", "changing_rooms"],
        "female_only_slots": ["07:00", "08:00", "09:00", "10:00"],  # More female slots
        "cancellation_rate": 0.10,
        "peak_hours": ["18:00", "19:00"],
        "maintenance_hours": ["23:00"],
    },
    "turf_004": {
        "name": "Urban Sports Hub",
        "capacity_types": {
            "11v11": {"max_players": 22, "size_m2": 9500},
            "7v7": {"max_players": 14, "size_m2": 4800},
            "5v5": {"max_players": 10, "size_m2": 2200}
        },
        "primary_sport": "football",
        "other_sports": ["futsal", "cricket"],
        "city": "Sonipat",
        "base_prices_per_slot": {
            "06:00": 1100, "07:00": 1100, "08:00": 1100,
            "09:00": 1100, "10:00": 1100, "11:00": 1100,
            "12:00": 1100, "13:00": 1100, "14:00": 1100,
            "15:00": 1100, "16:00": 1200, "17:00": 1200,
            "18:00": 1500, "19:00": 1500, "20:00": 1500,
            "21:00": 1500, "22:00": 1500
        },
        "facilities": ["parking", "changing_rooms", "floodlights"],
        "female_only_slots": [],
        "cancellation_rate": 0.10,
        "peak_hours": ["18:00", "19:00", "20:00"],
        "maintenance_hours": ["23:00"],
    },
    "turf_005": {
        "name": "Premier Cricket Arena",
        "capacity_types": {
            "20-overs": {"max_players": 22, "size_m2": 11500},
            "10-overs": {"max_players": 12, "size_m2": 7500}
        },
        "primary_sport": "cricket",
        "other_sports": ["tennis", "badminton"],
        "location": "Gurgaon",
        "base_prices_per_slot": {
            "06:00": 2100, "07:00": 2100, "08:00": 2600,
            "09:00": 2600, "10:00": 3100, "11:00": 3100,
            "12:00": 2600, "13:00": 2600, "14:00": 3100,
            "15:00": 3100, "16:00": 3300, "17:00": 3300,
            "18:00": 4600, "19:00": 5100, "20:00": 5100,
            "21:00": 4100, "22:00": 3600
        },
        "facilities": ["parking", "stands", "floodlights", "scoreboard", "medical_aid"],
        "female_only_slots": [],
        "cancellation_rate": 0.10,
        "peak_hours": ["18:00", "19:00", "20:00"],
        "maintenance_hours": ["23:00"],
    }
}

# Time slots mapping
TIME_SLOTS = ["06:00", "07:00", "08:00", "09:00", "10:00", "11:00",
              "12:00", "13:00", "14:00", "15:00", "16:00", "17:00",
              "18:00", "19:00", "20:00", "21:00", "22:00"]

TURF_IDS = list(TURFS.keys())
CAPACITY_TYPES = ["11v11", "7v7", "5v5"]  # Generic types

In [ ]:
def get_base_price(day, hour):
    """Returns the base price based on the day of the week and hour of the day."""
    # Base prices for each day of the week (0=Monday, 6=Sunday)
    base_prices = {
        0: 1200,  # Monday
        1: 1100,  # Tuesday
        2: 1200,  # Wednesday
        3: 1200,  # Thursday
        4: 1250,  # Friday
        5: 1400,  # Saturday
        6: 1400   # Sunday
    }
    
    # Hourly adjustment factors
    if 0 <= hour < 6:
        hour_factor = 0.8  # Off-peak hours
    elif 6 <= hour < 10:
        hour_factor = 1.0  # Morning hours
    elif 10 <= hour < 17:
        hour_factor = 0.8  # Afternoon hours
    else:
        hour_factor = 1.2  # Evening hours
    
    base_price = base_prices.get(day, 100) * hour_factor
    return base_price

In [ ]:
def get_booking_probability(base_price, dynamic_price, is_peak=False):
    """Returns the booking probability based on the price.
    Peak hours have lower price sensitivity (customers will pay more).
    Off-peak hours have higher price sensitivity.
    """
    price_diff = dynamic_price - base_price
    
    # Peak hours: customers are less price-sensitive (k is smaller = less steep decline)
    # Off-peak hours: customers are more price-sensitive (k is larger = steeper decline)
    if is_peak:
        k = 0.0015  # Low sensitivity at peak hours - customers will pay premium
    else:
        k = 0.008   # High sensitivity at off-peak - need lower prices
    
    prob = 1 / (1 + np.exp(k * price_diff))
    return prob

In [ ]:
def getTurfDetails(turf_id):
    """Mock function to get turf details."""
    # In a real scenario, this would fetch data from a database or API
    CITIES = ["New Delhi", "Mumbai", "Bangalore", "Chennai", "Kolkata", "Sonipat", "Gurgaon", "Imphal", "Gorakhpur"]
    TURFS = list(range(1, 10))  # Turf IDs from 1 to 10
    

In [ ]:
def get_demand_multiplier(turf_id, time_slot, day_of_week, capacity_type, is_female_slot=False):
    """
    Calculate demand multiplier based on multiple factors:
    - Location popularity
    - Time slot (peak vs off-peak)
    - Day of week (weekday vs weekend)
    - Sport type
    - Female slot availability
    """
    turf = TURFS[turf_id]
    base_multiplier = 1.0
    
    # Weekend boost (Saturday=5, Sunday=6)
    if day_of_week >= 5:
        base_multiplier *= 1.3
    
    # Peak hour boost
    if time_slot in turf["peak_hours"]:
        base_multiplier *= 1.8
    elif time_slot in ["07:00", "08:00", "21:00", "22:00"]:
        base_multiplier *= 0.6  # Off-peak penalty
    
    # Female-only slot boost (often higher demand for women-safe sports)
    if is_female_slot:
        base_multiplier *= 1.2
    
    # Capacity type affects demand (11v11 more popular than 5v5)
    if capacity_type == "11v11":
        base_multiplier *= 1.1
    elif capacity_type == "5v5":
        base_multiplier *= 0.8
    
    # Location multiplier
    location_mult = {
        "Delhi-North": 1.0,
        "Mumbai-Central": 1.2,
        "Bangalore-South": 0.9
    }
    location= turf.get("city", turf.get("location", "Unknown"))
    base_multiplier *= location_mult.get(location, 1.0)

    return base_multiplier

def get_booking_probability_turf(base_price, dynamic_price, demand_multiplier):
    """
    Calculate booking probability considering demand multiplier.
    Higher demand = customers more willing to pay premium.
    """
    price_diff = dynamic_price - base_price
    
    # Adjust price sensitivity based on demand
    if demand_multiplier > 1.5:  # High demand - low sensitivity
        k = 0.001
    elif demand_multiplier > 1.0:  # Medium demand
        k = 0.0015
    else:  # Low demand - high sensitivity
        k = 0.003
    
    prob = 1 / (1 + np.exp(k * price_diff))
    return prob

# Generate synthetic data with turf factors
# Generate synthetic data simulating the "XGBoost -> RL -> XGBoost" flow
data_with_turfs = []
num_rows = 150000

for i in range(num_rows):
    turf_id = np.random.choice(TURF_IDS)
    turf = TURFS[turf_id]
    time_slot = np.random.choice(TIME_SLOTS)
    day_of_week = np.random.randint(0, 7)  
    capacity_type = np.random.choice(list(turf["capacity_types"].keys()))
    
    is_female_slot = time_slot in turf["female_only_slots"] and np.random.rand() < 0.3
    base_price = turf["base_prices_per_slot"][time_slot]
    
    # 1. FIX THE BUG: Safely get location/city
    location_name = turf.get("city", turf.get("location", "Unknown"))
    
    # 2. SIMULATE XGBOOST (Baseline Probability)
    # We calculate the hidden demand multiplier, but we DO NOT give it to the RL agent.
    # Instead, we convert it into a baseline probability (what XGBoost would predict).
    hidden_demand_mult = get_demand_multiplier(turf_id, time_slot, day_of_week, capacity_type, is_female_slot)
    
    # If price doesn't change, base probability is driven purely by demand
    baseline_probability = get_booking_probability_turf(base_price, base_price, hidden_demand_mult)
    
    # Add noise so the RL agent doesn't overfit to a perfect mathematical formula
    baseline_probability = np.clip(baseline_probability + np.random.normal(0, 0.05), 0.05, 0.95)
    
    data_with_turfs.append({
        "turf_id": turf_id,
        "time_slot": time_slot,
        "day_of_week": day_of_week,
        "capacity_type": capacity_type,
        "is_female_slot": int(is_female_slot),
        "base_price": base_price,
        "baseline_probability": baseline_probability, # THE NEW STATE VARIABLE
        "hidden_demand_mult": hidden_demand_mult # Kept hidden to calculate the final reward
    })

df_turfs = pl.DataFrame(data_with_turfs)
print(f"Generated {len(df_turfs)} booking records with turf factors")
print("\nData sample:")
print(df_turfs.head(10))

In [ ]:
# %pip install gymnasium
import gymnasium as gym
from gymnasium import spaces


In [ ]:
class TurfDynamicPricingEnv(gym.Env):
    def __init__(self, data):
        super(TurfDynamicPricingEnv, self).__init__()
        self.data = data
        self.current_step = 0
        
        # Action space: price adjustment indices (7 price levels per turf)
        self.action_space = spaces.Discrete(7)  # 0-6 representing price multipliers
        
        # Observation space includes turf-specific factors
        self.observation_space = spaces.Dict({
            "turf_id_encoded": spaces.Discrete(len(TURF_IDS)),
            "time_slot_encoded": spaces.Discrete(len(TIME_SLOTS)),
            "day_of_week": spaces.Discrete(7),
            "capacity_type_encoded": spaces.Discrete(len(CAPACITY_TYPES)),
            "is_female_slot": spaces.Discrete(2),
            "base_price": spaces.Box(low=500, high=6000, shape=(1,), dtype=np.float32),
            "demand_multiplier": spaces.Box(low=0.5, high=2.0, shape=(1,), dtype=np.float32)
        })
        
        # Price multipliers: 0.9x, 0.95x, 1.0x, 1.05x, 1.1x, 1.2x, 1.3x
        self.price_multipliers = [0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.3]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        observation = self._get_observation()
        info = {}
        return (observation, info)

    def _get_observation(self):
        row = self.data[self.current_step]
        turf_id = row["turf_id"]
        time_slot = row["time_slot"]
        capacity_type = row["capacity_type"]
        
        turf_idx = TURF_IDS.index(turf_id)
        slot_idx = TIME_SLOTS.index(time_slot)
        capacity_idx = CAPACITY_TYPES.index(capacity_type) if capacity_type in CAPACITY_TYPES else 0
        
        return {
            "turf_id_encoded": turf_idx,
            "time_slot_encoded": slot_idx,
            "day_of_week": row["day_of_week"],
            "capacity_type_encoded": capacity_idx,
            "is_female_slot": row["is_female_slot"],
            "base_price": np.array([row["base_price"]], dtype=np.float32),
            "demand_multiplier": np.array([row["demand_multiplier"]], dtype=np.float32)
        }

    def step(self, action):
        row = self.data[self.current_step]
        
        # Calculate dynamic price based on base price and multiplier
        base_price = row["base_price"]
        multiplier = self.price_multipliers[action]
        chosen_price = int(base_price * multiplier)
        
        # Get true booking probability
        true_booking_prob = row["booking_probability"]
        demand_mult = row["demand_multiplier"]
        
        # Adjust booking probability based on our pricing decision
        # If we price higher than base, prob goes down; if lower, prob goes up
        price_delta = chosen_price - base_price
        price_impact = 1 / (1 + np.exp(0.002 * price_delta))
        
        final_booking_prob = min(true_booking_prob * price_impact * demand_mult, 1.0)
        was_booked = 1 if np.random.rand() < final_booking_prob else 0
        
        # Enhanced reward function for multiple objectives
        if was_booked:
            # Base revenue reward (normalized)
            revenue_reward = (chosen_price / 1000.0)
            
            # Boost for peak hours (18:00-20:00 typically peak)
            time_slot = row["time_slot"]
            peak_boost = 1.5 if time_slot in ["18:00", "19:00", "20:00"] else 1.0
            
            # Female slot priority (encourage affordable pricing for women)
            female_penalty = 0.95 if row["is_female_slot"] and chosen_price > base_price * 1.1 else 1.0
            
            # High demand slots - reward premium pricing
            demand_boost = 1.2 if demand_mult > 1.5 else 1.0
            
            reward = revenue_reward * peak_boost * female_penalty * demand_boost
        else:
            # Lost opportunity penalty
            reward = -0.3
        
        self.current_step += 1
        terminated = self.current_step >= len(self.data)
        truncated = False
        info = {
            "was_booked": was_booked,
            "price_chosen": chosen_price,
            "base_price": base_price,
            "revenue": chosen_price if was_booked else 0
        }
        observation = self._get_observation() if not terminated else {}
        return (observation, reward, terminated, truncated, info)

In [ ]:
# %pip install stable-baselines3[extra]
# %pip list stable-baselines3[extra] gymnasiums

In [ ]:
# %pip install --upgrade stable-baselines3 gymnasium

In [ ]:
from stable_baselines3.common.env_checker import check_env

env = TurfDynamicPricingEnv(data_with_turfs)
check_env(env)
print("✓ Turf Environment check passed!")

In [ ]:
from stable_baselines3 import PPO

env = TurfDynamicPricingEnv(data_with_turfs)
model = PPO(
    "MultiInputPolicy", 
    env, 
    verbose=1, 
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    ent_coef=0.01
)

print("Training Turf-Aware Dynamic Pricing Agent...")
model.learn(total_timesteps=150000)
model.save("turf_dynamic_pricing_model")
print("✓ Model training complete!")

In [ ]:
model = PPO.load("turf_dynamic_pricing_model")

def predict_price(turf_id, time_slot, day_of_week, capacity_type, is_female_slot=False):
    """Helper to predict optimal price for given scenario"""
    turf_idx = TURF_IDS.index(turf_id)
    slot_idx = TIME_SLOTS.index(time_slot)
    capacity_idx = CAPACITY_TYPES.index(capacity_type) if capacity_type in CAPACITY_TYPES else 0
    base_price = TURFS[turf_id]["base_prices_per_slot"][time_slot]
    
    # Approximate demand multiplier
    demand_mult = get_demand_multiplier(turf_id, time_slot, day_of_week, capacity_type, is_female_slot)
    
    obs = {
        "turf_id_encoded": turf_idx,
        "time_slot_encoded": slot_idx,
        "day_of_week": day_of_week,
        "capacity_type_encoded": capacity_idx,
        "is_female_slot": is_female_slot,
        "base_price": np.array([base_price], dtype=np.float32),
        "demand_multiplier": np.array([demand_mult], dtype=np.float32)
    }
    
    action, _ = model.predict(obs, deterministic=True)
    env = TurfDynamicPricingEnv(data_with_turfs)
    multiplier = env.price_multipliers[action]
    chosen_price = int(base_price * multiplier)
    
    return {
        "action": action,
        "multiplier": multiplier,
        "chosen_price": chosen_price,
        "base_price": base_price,
        "demand_multiplier": demand_mult
    }

print("\n" + "="*80)
print("TURF-AWARE DYNAMIC PRICING - TEST SCENARIOS")
print("="*80)

# Test 1: Elite Football - Peak Hour (High demand)
print("\n📍 TEST 1: Elite Football Arena - Saturday 8 PM (PEAK)")
result = predict_price("turf_001", "20:00", 5, "11v11", is_female_slot=False)
print(f"   Base Price: ₹{result['base_price']} | Demand: {result['demand_multiplier']:.2f}x")
print(f"   → AI Multiplier: {result['multiplier']:.2f}x")
print(f"   → PRICE: ₹{result['chosen_price']} (Action: {result['action']})")

# Test 2: Cricket Ground - Female Slot
print("\n👩 TEST 2: Cricket Grounds Pro - Female-Only Slot (Sunday 10 AM)")
result = predict_price("turf_002", "10:00", 6, "20-overs", is_female_slot=True)
print(f"   Base Price: ₹{result['base_price']} | Demand: {result['demand_multiplier']:.2f}x")
print(f"   → AI Multiplier: {result['multiplier']:.2f}x")
print(f"   → PRICE: ₹{result['chosen_price']} (Action: {result['action']})")

# Test 3: Multi-Sport - Off-Peak Weekday
print("\n🏟️  TEST 3: Multi-Sport Complex - Monday 7 AM (OFF-PEAK)")
result = predict_price("turf_003", "07:00", 0, "mixed", is_female_slot=False)
print(f"   Base Price: ₹{result['base_price']} | Demand: {result['demand_multiplier']:.2f}x")
print(f"   → AI Multiplier: {result['multiplier']:.2f}x")
print(f"   → PRICE: ₹{result['chosen_price']} (Action: {result['action']})")

# Test 4: Elite Football - Female Peak Slot
print("\n⚽ TEST 4: Elite Football - Female Peak (Saturday 9 PM)")
result = predict_price("turf_001", "21:00", 5, "7v7", is_female_slot=True)
print(f"   Base Price: ₹{result['base_price']} | Demand: {result['demand_multiplier']:.2f}x")
print(f"   → AI Multiplier: {result['multiplier']:.2f}x")
print(f"   → PRICE: ₹{result['chosen_price']} (Action: {result['action']})")

# Test 5: Cricket Ground - Peak Hour
print("\n🏏 TEST 5: Cricket Grounds Pro - Friday 7 PM (PEAK)")
result = predict_price("turf_002", "19:00", 4, "20-overs", is_female_slot=False)
print(f"   Base Price: ₹{result['base_price']} | Demand: {result['demand_multiplier']:.2f}x")
print(f"   → AI Multiplier: {result['multiplier']:.2f}x")
print(f"   → PRICE: ₹{result['chosen_price']} (Action: {result['action']})")

print("\n" + "="*80)

In [ ]:
def get_safe_ai_price(model, obs, possible_prices):
    """
    Wraps the AI model with human business rules (Guardrails).
    """
    # 1. Get the AI's "Greedy" Suggestion
    action, _ = model.predict(obs, deterministic=True)
    suggested_price = possible_prices[action]
    
    # 2. Apply Business Rules (The Guardrails)
    
    day = obs["day_of_week"]
    hour = obs["hour_of_day"]
    
    # Rule A: Strict Cap on Weekdays (Mon-Thu)
    if day < 4: # 0=Mon, 1=Tue, 2=Wed, 3=Thu
        MAX_WEEKDAY_PRICE = 3000
        if suggested_price > MAX_WEEKDAY_PRICE:
            print(f"⚠️ AI wanted {suggested_price}, but capped at {MAX_WEEKDAY_PRICE} (Weekday Rule)")
            return MAX_WEEKDAY_PRICE
            
    # Rule B: No Super-Surge before 6 PM
    if hour < 18:
        MAX_DAYTIME_PRICE = 3500
        if suggested_price > MAX_DAYTIME_PRICE:
             print(f"⚠️ AI wanted {suggested_price}, but capped at {MAX_DAYTIME_PRICE} (Daytime Rule)")
             return MAX_DAYTIME_PRICE

    # Rule C: Never go below Base Cost (Floor)
    MIN_PRICE = 2000
    if suggested_price < MIN_PRICE:
        return MIN_PRICE

    # If no rules broken, return the AI's price
    return suggested_price

# --- TESTING THE GUARDRAILS ---

# Setup the test observation for Saturday Peak (The one that was too high)
obs_saturday = {
    "day_of_week": 5,  # Saturday
    "hour_of_day": 20, # 8 PM
    "base_price": 1400
}

# Setup the test for Monday Evening (The one that was 3500)
obs_monday = {
    "day_of_week": 0, # Monday
    "hour_of_day": 21, # 9 PM
    "base_price": 1200
}

print("--- SATURDAY 8 PM ---")
base_price = get_base_price(obs_saturday["day_of_week"], obs_saturday["hour_of_day"])
print(f"Base Price for Saturday 8 PM: {base_price}")
final_price = get_safe_ai_price(model, obs_saturday, POSSIBLE_PRICES)
print(f"Final Customer Price: {final_price}")

base_price = get_base_price(obs_monday["day_of_week"], obs_monday["hour_of_day"])
print(f"Base Price for Monday 9 PM: {base_price}")
final_price = get_safe_ai_price(model, obs_monday, POSSIBLE_PRICES)
print(f"Final Customer Price: {final_price}")

In [ ]:
# RL vs ALTERNATIVES COMPARISON
print("\n" + "="*80)
print("ALGORITHM COMPARISON: RL vs Rule-Based vs Price Elasticity")
print("="*80)

# 1. HEURISTIC / RULE-BASED APPROACH
def heuristic_pricing(turf_id, time_slot, day_of_week, is_female_slot=False):
    """Simple rule-based pricing without ML"""
    turf = TURFS[turf_id]
    base = turf["base_prices_per_slot"][time_slot]
    
    # Simple rules:
    # - Peak hours: +30% premium
    # - Female slots: -10% discount for affordability
    # - Weekend: +20%
    
    price = base
    if time_slot in turf["peak_hours"]:
        price = int(price * 1.3)
    if day_of_week >= 5:
        price = int(price * 1.2)
    if is_female_slot:
        price = int(price * 0.9)
    
    return price

# 2. SIMPLE ELASTICITY MODEL (Price = Base * e^(-demand_sensitivity * quantity_sold))
def elasticity_pricing(turf_id, time_slot, day_of_week, booking_history_pct=0.75):
    """Price based on demand elasticity estimation"""
    turf = TURFS[turf_id]
    base = turf["base_prices_per_slot"][time_slot]
    
    # Estimate demand level from time/day
    demand_mult = get_demand_multiplier(turf_id, time_slot, day_of_week, "11v11", False)
    
    # Higher demand = can charge more
    if demand_mult > 1.5:
        return int(base * 1.25)
    elif demand_mult > 1.2:
        return int(base * 1.15)
    elif demand_mult < 0.8:
        return int(base * 0.85)
    else:
        return base

# Test scenarios for comparison
test_scenarios = [
    {"turf": "turf_001", "slot": "20:00", "day": 5, "female": False, "desc": "Elite Football - Peak Saturday"},
    {"turf": "turf_002", "slot": "10:00", "day": 6, "female": True, "desc": "Cricket - Female Morning"},
    {"turf": "turf_003", "slot": "07:00", "day": 0, "female": False, "desc": "Multi-Sport - Off-Peak Monday"},
    {"turf": "turf_001", "slot": "12:00", "day": 2, "female": False, "desc": "Elite Football - Noon Weekday"},
]

comparison_results = []

for scenario in test_scenarios:
    turf_id = scenario["turf"]
    time_slot = scenario["slot"]
    day = scenario["day"]
    is_female = scenario["female"]
    
    base_price = TURFS[turf_id]["base_prices_per_slot"][time_slot]
    
    # Get predictions from each method
    rl_result = predict_price(turf_id, time_slot, day, "11v11", is_female)
    rl_price = rl_result["chosen_price"]
    
    heuristic_price = heuristic_pricing(turf_id, time_slot, day, is_female)
    elasticity_price = elasticity_pricing(turf_id, time_slot, day)
    
    comparison_results.append({
        "scenario": scenario["desc"],
        "base": base_price,
        "rl": rl_price,
        "heuristic": heuristic_price,
        "elasticity": elasticity_price
    })
    
    print(f"\n{scenario['desc']}")
    print(f"  Base Price: ₹{base_price}")
    print(f"  RL Agent:      ₹{rl_price} ({((rl_price/base_price-1)*100):+.1f}%)")
    print(f"  Heuristic:     ₹{heuristic_price} ({((heuristic_price/base_price-1)*100):+.1f}%)")
    print(f"  Elasticity:    ₹{elasticity_price} ({((elasticity_price/base_price-1)*100):+.1f}%)")

print("\n" + "="*80)
print("ANALYSIS: When to use each approach:")
print("="*80)
print("""
✓ RL (PPO) is BEST for:
  - Complex multi-factor interactions (turf, capacity, location, sport, gender)
  - Learning non-obvious pricing patterns from historical data
  - Adapting over time as market conditions change
  - Joint optimization of revenue AND utilization
  - Handling rare scenarios automatically

✓ HEURISTIC RULES are BEST for:
  - Simple, explainable pricing (great for stakeholders)
  - Quick implementation with business domain knowledge
  - When you have clear, consistent business rules
  - No need for real-time adaptation
  ❌ Problem: Brittle, can't adapt to edge cases

✓ ELASTICITY MODELS are BEST for:
  - Quick baseline without ML complexity
  - Interpretable (easier to explain to CFO)
  - Good for initial MVP
  ❌ Problem: Assumes linear demand relationships (often not true)

🎯 RECOMMENDATION FOR YOUR SYSTEM:
  → Start with HEURISTIC rules + elasticity model (MVP)
  → Monitor performance metrics (revenue, utilization, cancellations)
  → Once you have >3-6 months of real data → Deploy RL (PPO)
  → Use RL predictions as "suggestions" with guardrails initially
  → Full RL control after validating >20% revenue improvement
""")